# 07 · Build NanoGPT: A Decoder-Only Transformer from Scratch

**Time:** about 90 minutes · **Prerequisites:** `03` and `06`

## Learning objectives
1. Assemble multi-head causal self-attention, a feed-forward network,
   residual connections, and LayerNorm into a GPT.
2. Train it on the notebook 03 corpus with a **leak-free validation split**,
   track validation perplexity, and keep the best checkpoint (**early stopping**).
3. **Look inside**: extract per-head attention maps and find heads with
   recognizable jobs.
4. Run a **positional-encoding ablation**, and learn the precise sense in
   which attention is order-blind.

```
Token IDs → token embedding + position embedding
          → [LayerNorm → causal multi-head attention → + residual
             LayerNorm → feed-forward             → + residual] × n_layer
          → LayerNorm → linear → next-token logits at every position
```

In [ ]:
import math

import numpy as np
import mlx.core as mx
import mlx.nn as nn
import mlx.optimizers as optim
import matplotlib.pyplot as plt
import seaborn as sns

from mlx_nlp_utils import (
    clean_holdout_slice, count_parameters, load_sample_corpus, make_train_step,
    print_device_info, sanity_check, scaled_dot_product_attention,
)

sns.set_style('whitegrid')
print_device_info()

## 1. Multi-head attention

One attention operation with 64-dimensional queries can express one
matching rule at a time. **Multi-head** attention splits the 64 features into
4 heads of 16, runs attention in each independently, and concatenates the
results, so different heads can learn different rules.

The fast path uses `mx.fast.scaled_dot_product_attention`, which never
materializes the $T \times T$ weight matrix. For inspection we add a
`capture` switch that uses the explicit implementation from notebook 06 and
stores the weights.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_head, n_embd, dropout=0.1):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)   # Q, K, V projections in one matmul
        self.c_proj = nn.Linear(n_embd, n_embd)       # mixes the heads back together
        self.resid_dropout = nn.Dropout(dropout)
        self.capture = False                          # set True to record attention weights
        self.last_weights = None

    def __call__(self, x, mask=None):
        B, T, C = x.shape
        q, k, v = mx.split(self.c_attn(x), 3, axis=-1)
        # (B, T, C) -> (B, n_head, T, head_dim)
        q, k, v = (t.reshape(B, T, self.n_head, self.head_dim).transpose(0, 2, 1, 3) for t in (q, k, v))

        if self.capture:
            allowed = mx.tril(mx.ones((T, T), dtype=mx.bool_)) if mask == "causal" else mask
            y, self.last_weights = scaled_dot_product_attention(q, k, v, allowed)
        else:
            y = mx.fast.scaled_dot_product_attention(q, k, v, scale=self.head_dim ** -0.5, mask=mask)

        y = y.transpose(0, 2, 1, 3).reshape(B, T, C)  # concatenate heads
        return self.resid_dropout(self.c_proj(y))

## 2. Feed-forward network

Attention moves information *between* positions; the feed-forward network
then transforms each position independently (expand 4×, GELU, project back).

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def __call__(self, x):
        return self.net(x)

## 3. The Transformer block

Each sub-layer is wrapped as `x = x + sublayer(LayerNorm(x))` ("pre-norm").
The **residual** path lets gradients flow straight through many layers;
**LayerNorm** keeps activations at a stable scale.

In [ ]:
class Block(nn.Module):
    def __init__(self, n_head, n_embd, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = MultiHeadAttention(n_head, n_embd, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffwd = FeedForward(n_embd, dropout)

    def __call__(self, x, mask=None):
        x = x + self.attn(self.ln1(x), mask)
        x = x + self.ffwd(self.ln2(x))
        return x

## 4. The GPT model

Positions use a **learned** embedding table (as in GPT-2) rather than
notebook 05's fixed sinusoids. `use_positions=False` switches it off for the
ablation in section 8.

In [ ]:
class GPT(nn.Module):
    def __init__(self, vocab_size, n_layer, n_head, n_embd, block_size, dropout=0.1, use_positions=True):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_head, n_embd, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.block_size = block_size
        self.use_positions = use_positions

    def __call__(self, idx, mask="causal"):
        B, T = idx.shape
        x = self.token_embedding(idx)
        if self.use_positions:
            x = x + self.position_embedding(mx.arange(T))
        for block in self.blocks.layers:
            x = block(x, mask)
        return self.lm_head(self.ln_f(x))              # (B, T, vocab): a prediction at every position

    def set_capture(self, on):
        for block in self.blocks.layers:
            block.attn.capture = on

## 5. Data: the notebook 03 corpus, split without leaks

Same corpus as notebook 03, so the perplexities can be compared. The
validation slice is the least-copied 10% of the text, chosen by
`clean_holdout_slice` (notebook 03 shows why the last 10% would be a leaky
choice). Training text is the rest; the single seam where the two training
pieces meet adds a handful of odd windows, which we accept for simplicity.

In [ ]:
corpus, _, _ = load_sample_corpus()
chars = sorted(set(corpus))
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}
vocab_size = len(chars)

start, end = clean_holdout_slice(corpus)
train_data = mx.array([char_to_idx[c] for c in corpus[:start] + corpus[end:]])
val_data = mx.array([char_to_idx[c] for c in corpus[start:end]])
print(f"Vocabulary {vocab_size} | train {len(train_data):,} chars | validation {len(val_data):,} chars")

def get_batch(data, block_size=32, batch_size=4):
    """Random windows; targets are the inputs shifted one character left."""
    if block_size < 1 or batch_size < 1 or len(data) <= block_size:
        raise ValueError("Need positive sizes and more tokens than block_size")
    starts = mx.random.randint(0, len(data) - block_size, (batch_size, 1))
    positions = starts + mx.arange(block_size)[None, :]
    return data[positions], data[positions + 1]

block_size = 64
xb, yb = get_batch(train_data, block_size, batch_size=2)
print("input :", repr(''.join(idx_to_char[int(i)] for i in xb[0][:30])))
print("target:", repr(''.join(idx_to_char[int(i)] for i in yb[0][:30])))

Unlike notebook 03, where each 5-character window produced one target, every
position in a GPT window is a training example: 64 predictions per window,
each seeing only the characters to its left (the causal mask).

## 6. Train with validation and early stopping

`val_perplexity` scores the whole validation slice in overlapping windows.
During training we evaluate every 25 steps and keep the parameters with the
best validation perplexity. MLX arrays are immutable and the optimizer
replaces them on each update, so holding on to `model.parameters()` is a
cheap checkpoint.

🤔 **Predict:** 2K training characters, ~100K parameters, 400 steps of 32
windows each. Where will the validation curve bottom out?

In [ ]:
def val_perplexity(model, data=val_data, block_size=block_size):
    model.eval()
    starts = range(0, len(data) - block_size - 1, block_size // 2)
    X = mx.stack([data[s:s + block_size] for s in starts])
    Y = mx.stack([data[s + 1:s + block_size + 1] for s in starts])
    loss = nn.losses.cross_entropy(model(X), Y, reduction='mean').item()
    model.train()
    return math.exp(loss)

def loss_fn(model, x, y):
    return nn.losses.cross_entropy(model(x), y, reduction='mean')

def train_gpt(model, steps, lr=3e-4, eval_every=25, seed=0):
    mx.random.seed(seed)
    optimizer = optim.AdamW(learning_rate=lr)
    model.train()
    step_fn, state = make_train_step(model, optimizer, loss_fn)
    history = {'step': [], 'train_ppl': [], 'val_ppl': []}
    best = (float('inf'), 0, model.parameters())
    for step in range(1, steps + 1):
        x, y = get_batch(train_data, block_size, batch_size=32)
        loss = step_fn(x, y)
        mx.eval(loss, state)
        if step % eval_every == 0 or step == steps:
            vp = val_perplexity(model)
            history['step'].append(step); history['train_ppl'].append(math.exp(loss.item())); history['val_ppl'].append(vp)
            if vp < best[0]:
                best = (vp, step, model.parameters())
    model.update(best[2])          # restore the best checkpoint
    model.eval()
    return history, best[0], best[1]

config = dict(vocab_size=vocab_size, n_layer=2, n_head=4, n_embd=64, block_size=block_size, dropout=0.1)
mx.random.seed(0)
model = GPT(**config)
print(f"Parameters: {count_parameters(model):,}")

train_steps = 400
history, best_ppl, best_step = train_gpt(model, train_steps)

plt.figure(figsize=(9, 4))
plt.plot(history['step'], history['train_ppl'], label='train (one batch)')
plt.plot(history['step'], history['val_ppl'], label='validation')
plt.axvline(best_step, color='gray', linestyle=':', label=f'best checkpoint (step {best_step})')
plt.yscale('log'); plt.xlabel('step'); plt.ylabel('perplexity (log)'); plt.legend()
plt.title('NanoGPT on 2K characters'); plt.show()
print(f"Best validation perplexity {best_ppl:.2f} at step {best_step}; final {history['val_ppl'][-1]:.2f}")
print(f"Uniform baseline: {vocab_size}")
sanity_check(best_ppl < vocab_size, "NanoGPT beats the uniform baseline")

**What happened.** Validation perplexity falls for only the first 50 or so
steps, then rises while training perplexity keeps dropping: the model starts memorizing
the training text. Early stopping restores the best checkpoint.

How good is the best value? Notebook 03 found about 14 for a character
trigram table and about 12 for the LSTM on the same validation slice. A
Transformer trained from scratch on 2,000 characters lands in the same range.
As notebook 05 found, its advantages appear with much more data.

## 7. Generate, and look inside

In [ ]:
def generate_text(model, start_str, max_new_tokens=100, temperature=1.0):
    """Sample one character at a time, cropping context to block_size."""
    if not start_str or temperature < 0 or max_new_tokens < 0:
        raise ValueError("Provide a nonempty seed and nonnegative temperature/token count")
    unknown = set(start_str) - set(char_to_idx)
    if unknown:
        raise ValueError(f"Seed contains characters not in the vocabulary: {sorted(unknown)}")
    model.eval()
    context = mx.array([char_to_idx[ch] for ch in start_str])[None]
    generated = start_str
    for _ in range(max_new_tokens):
        logits = model(context[:, -model.block_size:])[:, -1, :]
        next_idx = (mx.argmax(logits, axis=-1) if temperature == 0
                    else mx.random.categorical(logits / temperature))
        context = mx.concatenate([context, next_idx[:, None]], axis=1)
        generated += idx_to_char[next_idx.item()]
    return generated

mx.random.seed(0)
print(generate_text(model, "MLX is", max_new_tokens=200, temperature=0.8))

Each generated character reruns the model over the whole context. Real
implementations cache each layer's keys and values (a **KV cache**) so each
new token only computes its own row of attention.

### What do the heads attend to?

Switch on capture, run one sentence, and plot each head's attention map.

In [ ]:
text = "Arrays in MLX live in shared memory."
model.set_capture(True)
model(mx.array([[char_to_idx[c] for c in text]]))
model.set_capture(False)

labels = [c if c != ' ' else '_' for c in text]
n_layer, n_head = config['n_layer'], config['n_head']
fig, axes = plt.subplots(n_layer, n_head, figsize=(4 * n_head, 4 * n_layer))
prev_token_score = {}
for layer, block in enumerate(model.blocks.layers):
    w = np.array(block.attn.last_weights[0])          # (heads, T, T)
    for head in range(n_head):
        ax = axes[layer, head]
        ax.imshow(w[head], cmap='viridis', vmin=0, vmax=1)
        ax.set_xticks(range(len(text))); ax.set_xticklabels(labels, fontsize=6)
        ax.set_yticks(range(len(text))); ax.set_yticklabels(labels, fontsize=6)
        prev_token_score[layer, head] = float(np.mean(np.diagonal(w[head], offset=-1)))
        ax.set_title(f'layer {layer} head {head}\nweight on previous char: {prev_token_score[layer, head]:.2f}', fontsize=9)
plt.tight_layout(); plt.show()

best_head = max(prev_token_score, key=prev_token_score.get)
uniform = np.mean([1 / (t + 1) for t in range(1, len(text))])   # row t spreads over t+1 positions
print(f"Most 'previous-character' head: layer {best_head[0]} head {best_head[1]} "
      f"(average weight on the previous character: {prev_token_score[best_head]:.2f}; "
      f"uniform attention would give {uniform:.2f})")

**Reading the maps.** Rows are query positions, columns are the characters
they attend to; everything above the diagonal is zero (causal mask). Look for:

* **Previous-character heads**: a bright line just below the diagonal. With
  only 5–15 characters of useful context, attending to the last few characters
  is the most useful thing a head can do on this data.
* **Diffuse heads**: spread evenly over the past, acting like a running average.
* **Space / punctuation heads**: columns that light up at `_` or `.`, i.e.
  word boundaries.

The model we restored is the early-stopped checkpoint, only a few dozen steps
into training, so its heads are only weakly specialized: compare the best
previous-character score with the uniform value printed above. Patterns get
sharper with more training (Exercise 3), but on this corpus "more training"
also means memorization. With a model this small, patterns also vary between runs.
This kind of inspection is the starting point of **mechanistic
interpretability**, which studies the same patterns ("previous-token heads",
"induction heads") in large models.

## 8. Positional-encoding ablation

### The precise claim

Without positional information, self-attention **without a mask** is
*permutation-equivariant*: shuffle the input tokens and the outputs come out
shuffled the same way, with identical values. The model sees a bag of
tokens. Let's verify that numerically, then see what the causal mask changes.

In [ ]:
mx.random.seed(1)
no_pos = GPT(**{**config, 'use_positions': False}); no_pos.eval()
tokens = mx.array([[char_to_idx[c] for c in "dog bites man"]])
perm = mx.array(np.random.default_rng(0).permutation(tokens.shape[1]))
for mask, name in [(None, "no mask (encoder-style)"), ("causal", "causal mask (GPT-style)")]:
    out = no_pos(tokens, mask=mask)
    out_shuffled = no_pos(tokens[:, perm], mask=mask)
    gap = mx.abs(out[:, perm] - out_shuffled).max().item()
    print(f"{name:<26} max difference after un-shuffling: {gap:.2e}")

With no mask the difference is zero up to float rounding: "dog bites man"
and any shuffle of it are the same input to this model. With a causal mask
the difference is large: position $t$ can see exactly $t$ earlier tokens, so
**the mask itself leaks order information**. Research on "NoPE" (no positional
encoding) language models found that causal Transformers can learn to recover
position from this signal (Haviv et al., 2022; Kazemnejad et al., 2023).

### Does removing positions hurt this language model?

🤔 **Predict:** how much worse will validation perplexity be without
position embeddings?

In [ ]:
mx.random.seed(0)
no_pos_model = GPT(**{**config, 'use_positions': False})
no_pos_history, no_pos_best, no_pos_step = train_gpt(no_pos_model, train_steps)
print(f"Best validation perplexity with positions:    {best_ppl:.2f} (step {best_step})")
print(f"Best validation perplexity without positions: {no_pos_best:.2f} (step {no_pos_step})")

plt.figure(figsize=(9, 4))
plt.plot(history['step'], history['val_ppl'], label='with position embeddings')
plt.plot(no_pos_history['step'], no_pos_history['val_ppl'], label='without')
plt.yscale('log'); plt.xlabel('step'); plt.ylabel('validation perplexity'); plt.legend(); plt.show()

On this corpus the two end up close (here within roughly 10%), far from the
"cannot work at all" you might expect. Two reasons, both worth remembering:

1. The causal mask already provides order information (shown above).
2. The bottleneck is **data**: with 2K characters, the model overfits long
   before positional detail could matter.

Ablations answer "does this component matter **in this setting**?" A
component that is irrelevant at toy scale can be essential at scale, and the
reverse. Always state the setting with the result.

## Summary
* A GPT is embeddings → (attention + feed-forward with residuals and
  LayerNorm) × N → logits for the next token at every position.
* On tiny data it overfits within a hundred steps; validation tracking and
  early stopping are essential.
* Attention maps can be extracted and read; some heads learn interpretable roles.
* Without positions, unmasked attention is exactly order-blind, but a causal
  mask leaks order, so the ablation's effect depends on the setting.

## 🧠 Exercises

**1. Width vs data.** Set `n_embd=32` and then `n_embd=128`. 🤔 Predict which
has the better *best* validation perplexity.

<details><summary>What to look for</summary>

The larger model memorizes faster, so its best checkpoint arrives earlier.
Whether it is better at that point is an empirical question on 2K characters;
compare the spread across a couple of seeds before concluding anything.
</details>

**2. Context length.** Set `block_size = 8` (rebuild data windows and model).
How does best validation perplexity compare with notebook 03's 5-character LSTM?

<details><summary>What to look for</summary>

With 8 characters of context, the GPT and the LSTM see similar information,
so their best perplexities should be in the same range. This is a fairer
architecture comparison than one with different context lengths.
</details>

**3. Sharper heads.** Train a fresh model with `train_gpt(..., steps=400)`,
skip restoring the best checkpoint (comment out `model.update(best[2])`), and
redraw the head maps. What changed, and is the model better or worse?

<details><summary>What to look for</summary>

Heads become somewhat sharper (in our run the best previous-character score
rose from about 0.12 to 0.15), yet validation perplexity is far worse (about 36
vs 14): the extra structure partly encodes memorized training text. Clearer
attention maps are not evidence of a better model.
</details>

**4. Find a head's job.** Pick the head with the highest previous-character
score and zero its contribution (multiply its slice of `y` by 0 inside
`MultiHeadAttention` before `c_proj`). How much does validation perplexity rise?

<details><summary>What to look for</summary>

This is **ablating a head**, a basic interpretability experiment: if
perplexity rises noticeably, the model relied on that head. Small models
often have redundant heads, so removing one may cost less than you expect.
</details>

## ❓ FAQ

**Encoder vs decoder?** Encoders (BERT) use unmasked attention and suit
classification; decoders (GPT) use causal attention and generate text;
encoder–decoders (T5) map one sequence to another.

**What does LayerNorm do?** It rescales each position's features to zero
mean and unit variance (then applies a learned scale and shift), which keeps
deep stacks numerically stable.

**Next:** `08_Fine_Tuning_with_LoRA` adapts a *pretrained* GPT-style model
instead of training one from scratch.